## Research Question

#### Which entry signal better identifies profitable mean-reversion opportunities in cryptocurrency spread trading?

Baseline: Z-score mean-reversion pairs trading.

Alternative: Bollinger Bands + RSI on the same spread.

Extension: Use a machine learning model (e.g., logistic regression, random forest, or a small neural network) to predict whether a z-score signal is likely to revert successfully.

Individual ranges are unstable -- comparing spreads helps remove the market-wide movement

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import ccxt

In [2]:
# 1) 
# load price data
# align timestamps
# clean missing values
# compute log prices
# optionally resample to a fixed interval

In [3]:
exchange = ccxt.kraken()

eth = exchange.fetch_ohlcv("ETH/USD", timeframe="1h", limit=1000)
sol = exchange.fetch_ohlcv("SOL/USD", timeframe="1h", limit=1000)

In [4]:
eth = pd.DataFrame(eth, columns=["timestamp", "open", "high", "low", "close", "volume"])
eth["timestamp"] = pd.to_datetime(eth["timestamp"], unit="ms")
print(eth.head())

            timestamp     open     high      low    close       volume
0 2026-06-08 17:00:00  1682.00  1686.99  1674.88  1685.99  2527.810668
1 2026-06-08 18:00:00  1685.99  1691.64  1678.39  1684.08   799.954880
2 2026-06-08 19:00:00  1684.01  1689.81  1680.68  1684.45   852.796553
3 2026-06-08 20:00:00  1684.48  1688.92  1678.74  1688.32  1509.709997
4 2026-06-08 21:00:00  1688.99  1713.33  1688.99  1703.47  1075.594251


In [5]:
sol = pd.DataFrame(sol, columns=["timestamp", "open", "high", "low", "close", "volume"])
sol["timestamp"] = pd.to_datetime(sol["timestamp"], unit="ms")
print(eth.head())

            timestamp     open     high      low    close       volume
0 2026-06-08 17:00:00  1682.00  1686.99  1674.88  1685.99  2527.810668
1 2026-06-08 18:00:00  1685.99  1691.64  1678.39  1684.08   799.954880
2 2026-06-08 19:00:00  1684.01  1689.81  1680.68  1684.45   852.796553
3 2026-06-08 20:00:00  1684.48  1688.92  1678.74  1688.32  1509.709997
4 2026-06-08 21:00:00  1688.99  1713.33  1688.99  1703.47  1075.594251


In [6]:
# 2)
# Create one function that takes two price series and returns:
# hedge ratio β
# spread series
# maybe residuals if you use regression

# A simple first version can be:
# St = log(PA) + βlog(PB)

# This should be shared by both strategies.

In [7]:
import statsmodels.api as sm

In [8]:
X = np.log(eth[['close']])
# print(type(X))

Y = np.log(sol[['close']])
# print(X)
# print(Y)


In [9]:
X = sm.add_constant(X)

In [10]:
results = sm.OLS(Y, X).fit()

In [11]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  close   R-squared:                       0.243
Model:                            OLS   Adj. R-squared:                  0.241
Method:                 Least Squares   F-statistic:                     230.2
Date:                Wed, 08 Jul 2026   Prob (F-statistic):           2.55e-45
Time:                        10:54:11   Log-Likelihood:                 975.18
No. Observations:                 721   AIC:                            -1946.
Df Residuals:                     719   BIC:                            -1937.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.6818      0.393     -4.278      0.000      -2.454      -0.910
close          0.8030      0.053     15.174      0.000       0.699       0.907
==============================================================================
Omnibus:                     1554.550   Durbin-Watson:                   0.004
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               56.097
Skew:                           0.195   Prob(JB):                     6.59e-13
Kurtosis:                       1.690   Cond. No.                     1.27e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.27e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [12]:
results.params['const']


np.float64(-1.6818424025888703)

In [13]:
results.resid

0     -0.074475
1     -0.072674
2     -0.072702
3     -0.074693
4     -0.080829
         ...   
716    0.038448
717    0.039985
718    0.038462
719    0.039526
720    0.037952
Length: 721, dtype: float64

In [14]:
Y_est_resid = results.params['const'] + results.params['close']*X["close"] + results.resid
Y_est = results.params['const'] + results.params['close']*X["close"]

In [15]:
np.sum(abs(Y_est_resid-Y))

np.float64(0.0)

In [16]:
# 3) Implement the z-score strategy first

# rolling mean of spread
# rolling std of spread
# z-score calculation
# entry rule
# exit rule

# Example:

# enter long spread when z < -2
# enter short spread when z > 2
# exit when z returns near 0


In [17]:
spread = results.resid

In [18]:
rol_mean = spread.rolling(25).mean()
rol_std = spread.rolling(25).std()

In [19]:
Z_scores = (spread - rol_mean)/rol_std

In [20]:
res = pd.concat([pd.DataFrame(spread, columns=["spread"]), pd.DataFrame(Z_scores, columns=["Z"])], axis=1)

In [21]:
res['signal_eth'] =  np.where(
    res["Z"] < -1,
    1,
    np.where(res["Z"] > 1, -1, 0))
res["signal_sol"] = res['signal_eth']*-1

In [22]:
res

,spread,Z,signal_eth,signal_sol
0,-0.074475,NaN,0,0
1,-0.072674,NaN,0,0
2,-0.072702,NaN,0,0
3,-0.074693,NaN,0,0
4,-0.080829,NaN,0,0
...,...,...,...,...
716,0.038448,-1.224202,1,-1
717,0.039985,-1.025774,1,-1
718,0.038462,-1.055761,1,-1
719,0.039526,-0.905151,0,0


In [ ]:

# 4) Add the Bollinger Band version

# rolling mean
# rolling std
# upper/lower bands

# Example:

# long when spread crosses below lower band
# short when spread crosses above upper band
# exit at the center line or a fixed threshold


In [23]:
rol_mean.name = 'rol_mean'
rol_std.name = 'rol_std'

In [24]:
res = pd.concat([res, rol_mean, rol_std], axis=1)

In [25]:
res["UB"] = res['rol_mean'] + res['rol_std']
res["LB"] = res['rol_mean'] - res['rol_std']

In [26]:
res['signal_eth_bb'] = np.where(
    res["spread"] < res["LB"],
    1,
    np.where(res["spread"] > res["UB"], -1, 0)
)

In [27]:
res["signal_sol_bb"] = res["signal_eth_bb"]*-1

In [28]:
res

,spread,Z,signal_eth,signal_sol,rol_mean,rol_std,UB,LB,signal_eth_bb,signal_sol_bb
0,-0.074475,NaN,0,0,NaN,NaN,NaN,NaN,0,0
1,-0.072674,NaN,0,0,NaN,NaN,NaN,NaN,0,0
2,-0.072702,NaN,0,0,NaN,NaN,NaN,NaN,0,0
3,-0.074693,NaN,0,0,NaN,NaN,NaN,NaN,0,0
4,-0.080829,NaN,0,0,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...
716,0.038448,-1.224202,1,-1,0.055028,0.013544,0.068572,0.041484,1,-1
717,0.039985,-1.025774,1,-1,0.053840,0.013507,0.067346,0.040333,1,-1
718,0.038462,-1.055761,1,-1,0.052504,0.013300,0.065804,0.039204,1,-1
719,0.039526,-0.905151,0,0,0.051216,0.012916,0.064132,0.038301,0,0


In [29]:
# 5) Write one backtester that both strategies use

# Your backtester should handle:

# entering long/short spread positions
# closing positions
# transaction costs
# PnL calculation
# equity curve
# position sizing

In [30]:
eth_pos = pd.DataFrame({
    "mean_rev": res["signal_eth"],
    "bb": res["signal_eth_bb"]
})

In [32]:
sol_pos = pd.DataFrame({
    "mean_rev": res["signal_sol"],
    "bb": res["signal_sol_bb"]
})

In [34]:
beta = results.params["close"]

In [35]:
sol_pos = beta * sol_pos

In [39]:
eth_ret = (eth["close"] - eth["close"].shift(1)) / eth["close"].shift(1)

In [40]:
sol_ret = (sol["close"] - sol["close"].shift(1)) / sol["close"].shift(1)

In [45]:
strat_ret_mean_rev = (
    eth_pos["mean_rev"].shift(1)*eth_ret
    +
    sol_pos["mean_rev"].shift(1)*sol_ret
)


In [48]:
strat_ret_bb = (
    eth_pos["bb"].shift(1)*eth_ret
    +
    sol_pos["bb"].shift(1)*sol_ret
)

In [49]:
strat_both = pd.DataFrame({
    "mean_rev":strat_ret_mean_rev,
    "bb": strat_ret_bb
})

In [50]:
strat_both

,mean_rev,bb
0,NaN,NaN
1,0.000000,0.000000
2,0.000000,0.000000
3,0.000000,0.000000
4,0.000000,0.000000
...,...,...
716,-0.003220,-0.003220
717,-0.002937,-0.002937
718,-0.000119,-0.000119
719,0.001845,0.001845


In [ ]:
# 6) Add evaluation metrics

# Compare strategies using the same metrics:

# total return
# Sharpe ratio
# max drawdown
# win rate
# average trade return
# number of trades
# average holding time
# turnover

# If possible, also compare:

# how often they trade
# how sensitive they are to transaction costs
# how stable results are across different periods


In [51]:
def get_stats(strat_ret):
    stats = {}
    stats['SR'] = strat_ret.mean() / strat_ret.std() * np.sqrt(252)
    stats['ret'] = strat_ret.mean()*252
    stats['vol'] = strat_ret.std()*np.sqrt(252)
    stats = pd.Series(stats)
    return stats

In [52]:
metrics = get_stats(strat_both["mean_rev"])

In [53]:
metrics

SR     0.033886
ret    0.001473
vol    0.043480
dtype: float64

In [54]:
metrics = get_stats(strat_both["bb"])

In [55]:
metrics

SR     0.033886
ret    0.001473
vol    0.043480
dtype: float64

In [ ]:
# 7) Run one pair first, then expand

# Do not start with many pairs.

# Use one pair first, get the code working, then extend to:

# multiple crypto pairs
# different windows
# different thresholds
# out-of-sample testing